In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define dataset paths
export_path = f'{project_path}/data/labelstudio_export'  # <cat>/images and <cat>/labels
class_names = ['back-leg', 'body', 'front-leg', 'tail']

In [ ]:
# STEP 1: Split each cat's images into train, val and test
import random
import shutil
from pathlib import Path


def split_dataset(export_path, output_path, ratios, test_per_cat, seed=123):
    export_path, output_path = Path(export_path), Path(output_path)
    if output_path.exists():
        raise FileExistsError(f'{output_path} already exists, delete it first')
    rng = random.Random(seed)
    cats = sorted(d.name for d in export_path.iterdir() if d.is_dir())

    for cat in cats:
        images = sorted(p for p in (export_path / cat / 'images').iterdir()
                        if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        rng.shuffle(images)
        n = len(images)
        n_val = max(1, int(n * ratios[1])) if n >= 3 else 0
        n_test = max(1, int(n * ratios[2])) if n >= 3 else 0
        n_train = n - n_val - n_test
        splits = {'train': images[:n_train],
                  'val': images[n_train:n_train + n_val],
                  'test': images[n_train + n_val:]}

        for split, files in splits.items():
            dest = output_path / split / cat if (split == 'test' and test_per_cat) else output_path / split
            (dest / 'images').mkdir(parents=True, exist_ok=True)
            (dest / 'labels').mkdir(parents=True, exist_ok=True)
            for img in files:
                shutil.copy(img, dest / 'images' / img.name)
                label = export_path / cat / 'labels' / f'{img.stem}.txt'
                if label.exists():
                    shutil.copy(label, dest / 'labels' / label.name)

        print(f'{cat}: {n} images -> {n_train} train, {n_val} val, {n_test} test')

    # data.yaml for YOLO
    test = ''.join(f'\n  - test/{cat}/images' for cat in cats) if test_per_cat else ' test/images'
    names = ''.join(f'\n  {i}: {name}' for i, name in enumerate(class_names))
    (output_path / 'data.yaml').write_text(
        f'path: {output_path.resolve()}\ntrain: train/images\nval: val/images\ntest:{test}\nnames:{names}\n')

In [ ]:
# STEP 2: 70/15/15 split (YOLO model comparison)
split_dataset(export_path, f'{project_path}/data/yolo_70-15-15', (0.70, 0.15, 0.15), test_per_cat=False)

In [ ]:
# STEP 3: 25/15/60 split with a test folder per cat (FCM tests)
split_dataset(export_path, f'{project_path}/data/yolo_25-15-60', (0.25, 0.15, 0.60), test_per_cat=True)

In [ ]:
# STEP 4: Number of labels per body part in each split
from collections import Counter

data_path = Path(project_path, 'data')
for labels_path in sorted(data_path.glob('yolo_*/**/labels')):
    counts = Counter()
    for f in labels_path.glob('*.txt'):
        for line in f.read_text().splitlines():
            if line.strip():
                counts[class_names[int(float(line.split()[0]))]] += 1
    print(labels_path.parent.relative_to(data_path), dict(counts))